# Stat 175: Homework 5
## Matt Krasnow

In [ ]:
!pip install node2vec -q

## Exercise 1: Citation Network
### (a) Node Classification

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import networkx as nx
import urllib.request
import tarfile
import os

if not os.path.exists('/tmp/citeseer_data/citeseer'):
    urllib.request.urlretrieve(
        'https://linqs-data.soe.ucsc.edu/public/lbc/citeseer.tgz',
        '/tmp/citeseer.tgz'
    )
    with tarfile.open('/tmp/citeseer.tgz', 'r:gz') as tar:
        tar.extractall('/tmp/citeseer_data')

node_ids, features, labels = [], [], []
with open('/tmp/citeseer_data/citeseer/citeseer.content') as f:
    for line in f:
        parts = line.strip().split('\t')
        node_ids.append(parts[0])
        features.append([int(x) for x in parts[1:-1]])
        labels.append(parts[-1])

id_to_idx = {nid: i for i, nid in enumerate(node_ids)}
n = len(node_ids)
X = np.array(features, dtype=float)

label_set = sorted(set(labels))
label_to_int = {l: i for i, l in enumerate(label_set)}
y = np.array([label_to_int[l] for l in labels])

G = nx.Graph()
G.add_nodes_from(range(n))
with open('/tmp/citeseer_data/citeseer/citeseer.cites') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2 and parts[0] in id_to_idx and parts[1] in id_to_idx:
            u, v = id_to_idx[parts[0]], id_to_idx[parts[1]]
            if u != v:
                G.add_edge(u, v)

print(f'Citeseer: {n} nodes, {G.number_of_edges()} edges, {len(label_set)} classes')

In [ ]:
from node2vec import Node2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

node2vec = Node2Vec(G, dimensions=64, walk_length=20, num_walks=100, p=1, q=1, workers=1, seed=0)
model = node2vec.fit(window=10, min_count=1, batch_words=4)
Z = np.vstack([model.wv[str(v)] for v in range(n)])

idx = np.arange(n)
idx_train, idx_test = train_test_split(idx, test_size=0.3, random_state=0, stratify=y)
X_train, X_test = X[idx_train], X[idx_test]
Z_train, Z_test = Z[idx_train], Z[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
XZ_train = np.concatenate([X_train, Z_train], axis=1)
XZ_test = np.concatenate([X_test, Z_test], axis=1)

scaler_X = StandardScaler(with_mean=False)
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc = scaler_X.transform(X_test)
scaler_Z = StandardScaler()
Z_train_sc = scaler_Z.fit_transform(Z_train)
Z_test_sc = scaler_Z.transform(Z_test)
scaler_XZ = StandardScaler(with_mean=False)
XZ_train_sc = scaler_XZ.fit_transform(XZ_train)
XZ_test_sc = scaler_XZ.transform(XZ_test)

clf_X = LogisticRegression(max_iter=5000)
clf_X.fit(X_train_sc, y_train)
f1_X = f1_score(y_test, clf_X.predict(X_test_sc), average='macro')

clf_Z = LogisticRegression(max_iter=5000)
clf_Z.fit(Z_train_sc, y_train)
f1_Z = f1_score(y_test, clf_Z.predict(Z_test_sc), average='macro')

clf_XZ = LogisticRegression(max_iter=5000)
clf_XZ.fit(XZ_train_sc, y_train)
f1_XZ = f1_score(y_test, clf_XZ.predict(XZ_test_sc), average='macro')

print(f'Macro-F1 original features (X): {f1_X:.4f}')
print(f'Macro-F1 Node2vec embeddings (Z): {f1_Z:.4f}')
print(f'Macro-F1 combined (X|Z): {f1_XZ:.4f}')

### (b) Edge Prediction

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

edges = np.array(list(G.edges()))
num_nodes = G.number_of_nodes()
train_edges, test_edges = train_test_split(edges, test_size=0.3, random_state=0)

G_train = nx.Graph()
G_train.add_nodes_from(G.nodes())
G_train.add_edges_from(train_edges)

node2vec_ep = Node2Vec(G_train, dimensions=64, walk_length=20, num_walks=100, p=1, q=1, workers=1, seed=0)
model_ep = node2vec_ep.fit(window=10, min_count=1, batch_words=4)
Z_ep = np.vstack([model_ep.wv[str(v)] for v in range(num_nodes)])

all_edge_set = set()
for u, v in edges:
    a, b = min(u, v), max(u, v)
    all_edge_set.add((a, b))

rng = np.random.default_rng(0)

def sample_negative_edges(num_samples, forbidden, num_nodes, rng):
    neg, neg_set = [], set()
    while len(neg) < num_samples:
        u, v = rng.integers(0, num_nodes), rng.integers(0, num_nodes)
        if u == v: continue
        a, b = min(u, v), max(u, v)
        if (a,b) in forbidden or (a,b) in neg_set: continue
        neg.append((a, b)); neg_set.add((a, b))
    return np.array(neg)

neg_train = sample_negative_edges(len(train_edges), all_edge_set, num_nodes, rng)
neg_test = sample_negative_edges(len(test_edges), all_edge_set, num_nodes, rng)

def hadamard(pairs, Z): return np.array([Z[u]*Z[v] for u,v in pairs])

Xtr = np.vstack([hadamard(train_edges, Z_ep), hadamard(neg_train, Z_ep)])
ytr = np.hstack([np.ones(len(train_edges)), np.zeros(len(neg_train))])
Xte = np.vstack([hadamard(test_edges, Z_ep), hadamard(neg_test, Z_ep)])
yte = np.hstack([np.ones(len(test_edges)), np.zeros(len(neg_test))])

clf_ep = LogisticRegression(max_iter=5000)
clf_ep.fit(Xtr, ytr)
probs = clf_ep.predict_proba(Xte)[:, 1]

print(f'ROC-AUC: {roc_auc_score(yte, probs):.4f}')
print(f'Average Precision: {average_precision_score(yte, probs):.4f}')

## Exercise 3: Fairness
### (a) SBM Simulation

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n = 300
a = rng.integers(0, 2, size=n)

G_sbm = nx.Graph()
G_sbm.add_nodes_from(range(n))
for i in range(n):
    for j in range(i+1, n):
        p = 0.1 if a[i] == a[j] else 0.01
        if rng.random() < p:
            G_sbm.add_edge(i, j)

pos = nx.spring_layout(G_sbm, seed=0)
colors = ['tab:blue' if a[v]==0 else 'tab:orange' for v in G_sbm.nodes()]
plt.figure(figsize=(7,7))
nx.draw_networkx_nodes(G_sbm, pos, node_color=colors, node_size=30, alpha=0.9)
nx.draw_networkx_edges(G_sbm, pos, alpha=0.15)
plt.axis('off')
plt.title('SBM Graph colored by protected attribute')
plt.show()

### (b) Node2vec Embeddings

In [ ]:
from sklearn.decomposition import PCA

nv_sbm = Node2Vec(G_sbm, dimensions=32, walk_length=20, num_walks=100, p=1, q=1, workers=1, seed=0)
m_sbm = nv_sbm.fit(window=10, min_count=1, batch_words=4)
Z_sbm = np.vstack([m_sbm.wv[str(v)] for v in range(n)])

Z2 = PCA(n_components=2).fit_transform(Z_sbm)
plt.figure(figsize=(7,6))
for group, color in [(0,'tab:blue'),(1,'tab:orange')]:
    idx = np.where(a==group)[0]
    plt.scatter(Z2[idx,0], Z2[idx,1], s=20, alpha=0.8, label=f'a={group}')
plt.legend(); plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.title('Node2vec embeddings (PCA) by protected attribute')
plt.show()

### (c) Regression Fairness

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

p_feat = 5
X_feat = rng.normal(size=(n, p_feat))
beta = rng.normal(size=p_feat)
beta2 = 1.0
eps = rng.normal(scale=0.5, size=n)
y_reg = X_feat @ beta + beta2 * a + eps

idx_all = np.arange(n)
train_idx = rng.choice(idx_all, size=int(0.6*n), replace=False)
test_idx = np.setdiff1d(idx_all, train_idx)

XZ_reg = np.concatenate([X_feat, Z_sbm], axis=1)
reg = LinearRegression()
reg.fit(XZ_reg[train_idx], y_reg[train_idx])
yhat = reg.predict(XZ_reg[test_idx])

print(f'Test MSE: {mean_squared_error(y_reg[test_idx], yhat):.4f}')
print(f'Correlation(prediction, a): {np.corrcoef(yhat, a[test_idx])[0,1]:.4f}')
print(f'Mean pred a=0: {yhat[a[test_idx]==0].mean():.4f}')
print(f'Mean pred a=1: {yhat[a[test_idx]==1].mean():.4f}')